# Stage 5 — Model Deployment with Flask + ngrok
**Input :** `model.pkl`, `feature_names.pkl`, `le_*.pkl` (from Stage 4)  
**What this stage does :** Wraps the trained model in a Flask REST API and exposes it publicly via ngrok  
**Output :** A live public URL you can call from anywhere (Postman, Selenium scraper, frontend, etc.)

---

## 0. Install Dependencies

In [4]:
# Run this cell once
!pip install flask pyngrok joblib scikit-learn pandas numpy --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports & Load Saved Artifacts

In [5]:
import joblib
import numpy as np
import pandas as pd

# Load everything saved in Stage 4
model        = joblib.load('model.pkl')
FEATURES     = joblib.load('feature_names.pkl')
le_fuel      = joblib.load('le_fuel.pkl')
le_brand     = joblib.load('le_brand.pkl')
le_model_enc = joblib.load('le_model.pkl')
le_trans_sub = joblib.load('le_trans_sub.pkl')

print(f'Model loaded   : {type(model).__name__}')
print(f'Features count : {len(FEATURES)}')
print(f'Features       : {FEATURES}')

Model loaded   : GradientBoostingRegressor
Features count : 23
Features       : ['Brand', 'Model', 'Make Year', 'Car Age (years)', 'Kilometres Driven', 'km_per_year', 'Fuel Type', 'Transmission Type', 'Transmission Subtype', 'BS Norm', 'Number of Previous Owners', 'Insurance Type', 'Overall Quality Score', 'Engine Score', 'Systems Score', 'Interior Score', 'Exterior Score', 'Wear Score', 'Composite_Score', 'Value_Score', 'Depreciation_Category', 'Meter Tampered', 'Flooded']


## 2. Encoding Helper (same as Stage 4)

In [6]:
def safe_encode(le, val):
    """Label-encode a value; return 0 if unseen label."""
    return int(le.transform([val])[0]) if val in le.classes_ else 0


def build_feature_row(data: dict) -> pd.DataFrame:
    """
    Converts a raw input dict (from the API request)
    into a single-row DataFrame with the exact FEATURES order.
    """
    owner_map = {'1st Owner': 1, '2nd Owner': 2,
                 '3rd Owner': 3, '4th Owner': 4}
    dep_map   = {'Low': 1, 'Medium': 2, 'High': 3, 'Very High': 4}
    bs_map    = {'Unknown': 0, 'BSIV': 1, 'BSVI': 2}

    car_age   = int(data.get('car_age', 3))
    km        = int(data.get('km_driven', 50000))
    make_year = int(data.get('make_year', 2021))

    scores = [
        float(data.get('overall_quality', 8.0)),
        float(data.get('engine_score',    8.0)),
        float(data.get('systems_score',   8.0)),
        float(data.get('interior_score',  8.0)),
        float(data.get('exterior_score',  8.0)),
        float(data.get('wear_score',      8.0)),
    ]
    composite = round(float(np.mean(scores)), 3)
    km_per_yr = km // max(car_age, 1)

    row = {
        'Brand':                     safe_encode(le_brand,     data.get('brand', '')),
        'Model':                     safe_encode(le_model_enc, data.get('model', '')),
        'Make Year':                 make_year,
        'Car Age (years)':           car_age,
        'Kilometres Driven':         km,
        'km_per_year':               km_per_yr,
        'Fuel Type':                 safe_encode(le_fuel,      data.get('fuel_type', 'Petrol')),
        'Transmission Type':         0 if data.get('transmission', 'Manual') == 'Manual' else 1,
        'Transmission Subtype':      safe_encode(le_trans_sub, data.get('trans_sub', 'Regular')),
        'BS Norm':                   bs_map.get(data.get('bs_norm', 'Unknown'), 0),
        'Number of Previous Owners': owner_map.get(data.get('owners', '1st Owner'), 1),
        'Insurance Type':            0 if data.get('insurance', 'Third Party') == 'Third Party' else 1,
        'Overall Quality Score':     scores[0],
        'Engine Score':              scores[1],
        'Systems Score':             scores[2],
        'Interior Score':            scores[3],
        'Exterior Score':            scores[4],
        'Wear Score':                scores[5],
        'Composite_Score':           composite,
        'Value_Score':               float(data.get('value_score', 7.0)),
        'Depreciation_Category':     dep_map.get(data.get('depreciation', 'Medium'), 2),
        'Meter Tampered':            0 if data.get('meter_tampered', 'No') == 'No' else 1,
        'Flooded':                   0 if data.get('flooded', 'No') == 'No' else 1,
    }
    return pd.DataFrame([row])[FEATURES]


# Quick sanity check
test_row = build_feature_row({
    'brand': 'Tata', 'model': 'Nexon', 'make_year': 2021, 'car_age': 4,
    'km_driven': 50000, 'fuel_type': 'Petrol', 'transmission': 'Manual',
    'trans_sub': 'Regular', 'bs_norm': 'BSVI', 'owners': '1st Owner',
    'insurance': 'Comprehensive', 'overall_quality': 8.5, 'engine_score': 9.0,
    'systems_score': 8.8, 'interior_score': 8.2, 'exterior_score': 8.5,
    'wear_score': 8.3, 'value_score': 7.2, 'depreciation': 'High',
    'meter_tampered': 'No', 'flooded': 'No'
})
test_price = model.predict(test_row)[0]
print(f'Sanity check prediction: ₹{test_price:,.0f}  ({test_price/1e5:.2f} Lakhs)')
print('✅ Encoder + model pipeline working')

Sanity check prediction: ₹642,387  (6.42 Lakhs)
✅ Encoder + model pipeline working


## 3. Flask App

In [7]:
from flask import Flask, request, jsonify

app = Flask(__name__)


# ── GET /health ───────────────────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        'status':     'ok',
        'model':      type(model).__name__,
        'n_features': len(FEATURES),
    })


# ── POST /predict ─────────────────────────────────────────────────────────────
@app.route('/predict', methods=['POST'])
def predict():
    """
    Accepts JSON body with car features → returns predicted price.

    Required fields (all others have safe defaults):
      brand, model, make_year, car_age, km_driven,
      fuel_type, transmission, trans_sub, bs_norm,
      owners, insurance, overall_quality, engine_score,
      systems_score, interior_score, exterior_score,
      wear_score, value_score, depreciation,
      meter_tampered, flooded
    """
    data = request.get_json(force=True)
    if not data:
        return jsonify({'error': 'Send a JSON body with car features.'}), 400

    try:
        X     = build_feature_row(data)
        price = float(model.predict(X)[0])
        lakh  = round(price / 1e5, 2)

        return jsonify({
            'predicted_price_inr':   round(price),
            'predicted_price_lakh':  lakh,
            'formatted':             f'₹{price:,.0f}  ({lakh} Lakhs)',
            'model_used':            type(model).__name__,
            'features_used':         len(FEATURES),
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── POST /predict-batch ───────────────────────────────────────────────────────
@app.route('/predict-batch', methods=['POST'])
def predict_batch():
    """
    Accepts a JSON array of car objects → returns predictions for all.

    Body: [ {car1}, {car2}, ... ]
    """
    data = request.get_json(force=True)
    if not isinstance(data, list):
        return jsonify({'error': 'Send a JSON array of car objects.'}), 400

    try:
        predictions = []
        for i, car in enumerate(data):
            X     = build_feature_row(car)
            price = float(model.predict(X)[0])
            lakh  = round(price / 1e5, 2)
            predictions.append({
                'index':                 i,
                'predicted_price_inr':   round(price),
                'predicted_price_lakh':  lakh,
                'formatted':             f'₹{price:,.0f}  ({lakh} Lakhs)',
            })
        return jsonify({'count': len(predictions), 'predictions': predictions})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


print('✅ Flask app defined with 3 endpoints:')
print('   GET  /health')
print('   POST /predict')
print('   POST /predict-batch')

✅ Flask app defined with 3 endpoints:
   GET  /health
   POST /predict
   POST /predict-batch


## 4. Start ngrok Tunnel + Flask Server

> **Get your free ngrok token at:** https://dashboard.ngrok.com/get-started/your-authtoken  
> Paste it into `NGROK_AUTH_TOKEN` below.

In [8]:
from pyngrok import ngrok, conf
import threading

# ── 1. Set your ngrok auth token ──────────────────────────────────────────────
NGROK_AUTH_TOKEN = '3Cl2E13GANHC5HLV6MCA9SEtgYc_7Rd5aieZ79tmmnHX8nqrg'   # ← replace this
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# ── 2. Kill any existing ngrok tunnels (clean start) ─────────────────────────
ngrok.kill()

# ── 3. Open tunnel on port 5000 ───────────────────────────────────────────────
PORT       = 5000
public_url = ngrok.connect(PORT).public_url

print('\n' + '='*60)
print(f'  🚀  PUBLIC URL      : {public_url}')
print(f'  🏥  Health check    : {public_url}/health')
print(f'  🔢  Single predict  : POST {public_url}/predict')
print(f'  📦  Batch predict   : POST {public_url}/predict-batch')
print('='*60 + '\n')

# ── 4. Run Flask in a background thread (so Jupyter stays interactive) ────────
def run_flask():
    app.run(port=PORT, use_reloader=False, debug=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print('✅ Flask server running in background thread')


  🚀  PUBLIC URL      : https://humming-shadow-contend.ngrok-free.dev
  🏥  Health check    : https://humming-shadow-contend.ngrok-free.dev/health
  🔢  Single predict  : POST https://humming-shadow-contend.ngrok-free.dev/predict
  📦  Batch predict   : POST https://humming-shadow-contend.ngrok-free.dev/predict-batch

 * Serving Flask app '__main__'
✅ Flask server running in background thread
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


## 5. Test the Live API (from inside the notebook)

In [9]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import requests
import time
time.sleep(2)   # give Flask a moment to start

BASE = public_url   # uses the ngrok URL printed above

# ── Health check ──────────────────────────────────────────────────────────────
r = requests.get(f'{BASE}/health')
print('Health check response:', r.json())

127.0.0.1 - - [25/Apr/2026 22:15:57] "GET /health HTTP/1.1" 200 -


Health check response: {'model': 'GradientBoostingRegressor', 'n_features': 23, 'status': 'ok'}


In [11]:
# ── Single prediction ─────────────────────────────────────────────────────────
car = {
    'brand': 'Tata', 'model': 'Nexon', 'make_year': 2021, 'car_age': 4,
    'km_driven': 50000, 'fuel_type': 'Petrol', 'transmission': 'Manual',
    'trans_sub': 'Regular', 'bs_norm': 'BSVI', 'owners': '1st Owner',
    'insurance': 'Comprehensive', 'overall_quality': 8.5, 'engine_score': 9.0,
    'systems_score': 8.8, 'interior_score': 8.2, 'exterior_score': 8.5,
    'wear_score': 8.3, 'value_score': 7.2, 'depreciation': 'High',
    'meter_tampered': 'No', 'flooded': 'No'
}

r = requests.post(f'{BASE}/predict', json=car)
print('Single prediction:')
print(r.json())

127.0.0.1 - - [25/Apr/2026 22:16:02] "POST /predict HTTP/1.1" 200 -


Single prediction:
{'features_used': 23, 'formatted': '₹642,387  (6.42 Lakhs)', 'model_used': 'GradientBoostingRegressor', 'predicted_price_inr': 642387, 'predicted_price_lakh': 6.42}


In [12]:
# ── Batch prediction — all 5 college SUV profiles ─────────────────────────────
batch = [
    {'brand':'Tata',          'model':'Nexon',  'make_year':2021, 'car_age':4,
     'km_driven':50000,  'fuel_type':'Petrol',  'transmission':'Manual',
     'trans_sub':'Regular', 'bs_norm':'BSVI', 'owners':'1st Owner',
     'insurance':'Comprehensive', 'overall_quality':8.5, 'engine_score':9.0,
     'systems_score':8.8, 'interior_score':8.2, 'exterior_score':8.5,
     'wear_score':8.3, 'value_score':7.2, 'depreciation':'High',
     'meter_tampered':'No', 'flooded':'No'},

    {'brand':'Hyundai',       'model':'Creta',  'make_year':2020, 'car_age':5,
     'km_driven':72000,  'fuel_type':'Diesel',  'transmission':'Manual',
     'trans_sub':'Regular', 'bs_norm':'BSVI', 'owners':'1st Owner',
     'insurance':'Comprehensive', 'overall_quality':8.2, 'engine_score':8.8,
     'systems_score':8.5, 'interior_score':7.9, 'exterior_score':8.0,
     'wear_score':7.8, 'value_score':6.9, 'depreciation':'High',
     'meter_tampered':'No', 'flooded':'No'},

    {'brand':'Kia',           'model':'Seltos', 'make_year':2022, 'car_age':3,
     'km_driven':35000,  'fuel_type':'Petrol',  'transmission':'Automatic',
     'trans_sub':'DCT',     'bs_norm':'BSVI', 'owners':'1st Owner',
     'insurance':'Comprehensive', 'overall_quality':9.0, 'engine_score':9.2,
     'systems_score':9.0, 'interior_score':8.8, 'exterior_score':8.9,
     'wear_score':9.1, 'value_score':7.8, 'depreciation':'Medium',
     'meter_tampered':'No', 'flooded':'No'},

    {'brand':'Maruti Suzuki', 'model':'Brezza', 'make_year':2019, 'car_age':6,
     'km_driven':85000,  'fuel_type':'Petrol',  'transmission':'Manual',
     'trans_sub':'Regular', 'bs_norm':'BSIV', 'owners':'2nd Owner',
     'insurance':'Third Party', 'overall_quality':7.5, 'engine_score':7.8,
     'systems_score':7.6, 'interior_score':7.0, 'exterior_score':7.2,
     'wear_score':7.0, 'value_score':6.2, 'depreciation':'Very High',
     'meter_tampered':'No', 'flooded':'No'},

    {'brand':'Mahindra',      'model':'XUV',    'make_year':2018, 'car_age':7,
     'km_driven':110000, 'fuel_type':'Diesel',  'transmission':'Manual',
     'trans_sub':'Regular', 'bs_norm':'BSIV', 'owners':'2nd Owner',
     'insurance':'Third Party', 'overall_quality':7.0, 'engine_score':7.5,
     'systems_score':7.2, 'interior_score':6.8, 'exterior_score':6.9,
     'wear_score':6.5, 'value_score':5.8, 'depreciation':'Very High',
     'meter_tampered':'No', 'flooded':'No'},
]

r = requests.post(f'{BASE}/predict-batch', json=batch)
resp = r.json()
print(f"\nBatch predictions ({resp['count']} cars):")
for p in resp['predictions']:
    print(f"  [{p['index']+1}] {p['formatted']}")

127.0.0.1 - - [25/Apr/2026 22:16:07] "POST /predict-batch HTTP/1.1" 200 -



Batch predictions (5 cars):
  [1] ₹642,387  (6.42 Lakhs)
  [2] ₹919,141  (9.19 Lakhs)
  [3] ₹1,077,670  (10.78 Lakhs)
  [4] ₹466,798  (4.67 Lakhs)
  [5] ₹554,377  (5.54 Lakhs)


## 6. Use the API in Your Selenium Scraper

Copy the `public_url` printed above and paste it into your scraper.  
For each scraped car, call `/predict` to get a live price estimate.

In [ ]:
# ── Snippet to paste into your Selenium scraper ───────────────────────────────
SCRAPER_SNIPPET = f'''
import requests

NGROK_URL = "{public_url}"   # your live URL

def predict_price(car_dict):
    """Call the prediction API for one scraped car."""
    r = requests.post(f"{{NGROK_URL}}/predict", json=car_dict, timeout=10)
    return r.json().get("predicted_price_inr", None)

# Inside your scraping loop:
# price = predict_price({{
#     "brand": name_scraped, "car_age": age_scraped, ...
# }})
'''
print(SCRAPER_SNIPPET)


import requests

NGROK_URL = "https://humming-shadow-contend.ngrok-free.dev"   # your live URL

def predict_price(car_dict):
    """Call the prediction API for one scraped car."""
    r = requests.post(f"{NGROK_URL}/predict", json=car_dict, timeout=10)
    return r.json().get("predicted_price_inr", None)

# Inside your scraping loop:
# price = predict_price({
#     "brand": name_scraped, "car_age": age_scraped, ...
# })



127.0.0.1 - - [25/Apr/2026 22:16:34] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [25/Apr/2026 22:16:35] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [25/Apr/2026 22:18:31] "GET / HTTP/1.1" 404 -
t=2026-04-25T22:51:41+0530 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=8e0eda5a7b5a err="read tcp 192.168.0.136:53224->3.12.62.205:443: wsarecv: A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond."
t=2026-04-25T22:51:41+0530 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=36951f023e72 clientid=77683a3c54d15da372c497a3152f8c6d


## 7. curl Examples (run from Terminal)

After copying your ngrok URL, test the API directly from a terminal:

In [ ]:
print(f"""
# Health check
curl {public_url}/health

# Single prediction
curl -X POST {public_url}/predict \\
     -H "Content-Type: application/json" \\
     -d '{{
       "brand": "Tata",
       "model": "Nexon",
       "make_year": 2021,
       "car_age": 4,
       "km_driven": 50000,
       "fuel_type": "Petrol",
       "transmission": "Manual",
       "trans_sub": "Regular",
       "bs_norm": "BSVI",
       "owners": "1st Owner",
       "insurance": "Comprehensive",
       "overall_quality": 8.5,
       "engine_score": 9.0,
       "systems_score": 8.8,
       "interior_score": 8.2,
       "exterior_score": 8.5,
       "wear_score": 8.3,
       "value_score": 7.2,
       "depreciation": "High",
       "meter_tampered": "No",
       "flooded": "No"
     }}'
""")

## 8. Stop the Server (when done)

In [ ]:
# Run this cell to close the ngrok tunnel
ngrok.kill()
print('✅ ngrok tunnel closed')

---
## Stage 5 Summary

| What | Detail |
|---|---|
| Model loaded | `model.pkl` (best from Stage 4) |
| API framework | Flask |
| Public tunnel | ngrok (free tier) |
| Endpoints | `GET /health`, `POST /predict`, `POST /predict-batch` |
| Input | JSON with 21 car features |
| Output | `predicted_price_inr`, `predicted_price_lakh`, `formatted` |
| Next step | Stage 6 — Frontend UI or Streamlit dashboard |